# LeetCode 4043 — Count Rotations With Exactly K Equal Adjacent Pairs bằng Python

Link đề: https://leetcode.com/problems/count-rotations-with-exactly-k-equal-adjacent-pairs/ · Độ khó: **Easy** · Chủ đề: String, Enumeration, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def countRotations(self, s: str, k: int) -> int:
        ...
```

Ghi chú: đề và các ví dụ lấy từ đề gốc (Weekly Contest 518, câu 1); mọi đáp án trong notebook đều được kiểm chứng bằng code, kể cả bằng một lời giải độc lập viết riêng cho phần test.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có chuỗi `s` độ dài `n`; **xoay vòng** chuỗi bằng cách chuyển một tiền tố độ dài `0..n-1` xuống cuối; mỗi phép xoay có một **điểm số** = số cặp ký tự **liền kề giống nhau**; đếm xem có bao nhiêu phép xoay có điểm đúng bằng `k`.
2. Làm được cách **brute force**: dựng từng chuỗi xoay bằng cắt lát (`s[p:] + s[:p]`) rồi đếm cặp giống nhau, độ phức tạp `O(n²)`.
3. Nhận ra **bước đột phá**: coi chuỗi là một **vòng tròn** có đúng `n` cặp liền kề; mỗi phép xoay chỉ **làm mất đúng một cặp** của vòng tròn đó, nên `điểm = T − cyc[(p−1) mod n]`, với `T` là tổng số cặp giống nhau trên vòng.
4. Đi từ `O(n²)` → `O(n)` bằng nhiều cách: công thức vòng tròn, cập nhật trượt `O(1)` mỗi lần xoay, tổng tiền tố trên `s + s`, và công thức đếm đóng.
5. Tránh các lỗi kinh điển: quên cặp nối đầu–cuối, đếm cặp ở chỗ "khâu" của phép xoay, gộp các phép xoay cho ra chuỗi trùng nhau, và `%` sai khi chỉ số âm.

## 2. Đề bài & ví dụ

Cho chuỗi `s` độ dài `n` và một số nguyên `k`.

Một **phép xoay vòng** (cyclic rotation) của `s` được tạo bằng cách chọn một **tiền tố** của `s` có độ dài từ `0` đến `n - 1` (bao gồm cả hai đầu mút) rồi chuyển tiền tố đó xuống **cuối chuỗi**, giữ nguyên thứ tự các ký tự.

Với **mỗi** phép xoay, **điểm số (score)** là số chỉ số `i` thỏa `0 <= i < n - 1` mà ký tự ở vị trí `i` và `i + 1` **bằng nhau**.

Trả về **số phép xoay** của `s` có điểm số bằng `k`.

**Ví dụ 1**

- Input: `s = "aab"`, `k = 1`
- Output: `2`
- Giải thích: các phép xoay là `"aab"` (cặp vị trí 0,1 bằng nhau: điểm 1), `"aba"` (không cặp nào bằng nhau: điểm 0), `"baa"` (cặp vị trí 1,2 bằng nhau: điểm 1). Có 2 phép xoay điểm bằng `k = 1`.

**Ví dụ 2**

- Input: `s = "abca"`, `k = 0`
- Output: `1`
- Giải thích: các phép xoay là `"abca"` (điểm 0), `"bcaa"` (điểm 1), `"caab"` (điểm 1), `"aabc"` (điểm 1). Chỉ có 1 phép xoay có điểm bằng `0`.

**Ràng buộc:** `2 <= n == s.length <= 100`; `s` chỉ gồm chữ cái tiếng Anh thường; `0 <= k <= n - 1`.

> Lưu ý: ta đếm theo **mỗi lần chọn độ dài tiền tố** `p = 0, 1, ..., n-1` (tổng cộng đúng `n` phép xoay), kể cả khi hai phép xoay cho ra cùng một chuỗi (ví dụ `"aa"`). Cách đếm này khớp với lời giải gốc.

In [1]:
from typing import List
import random
from collections import Counter
from itertools import accumulate, pairwise

examples = [
    ("aab", 1, 2),
    ("abca", 0, 1),
]
for s, k, expected in examples:
    print(f"s={s!r}, k={k} -> kỳ vọng {expected}")

s='aab', k=1 -> kỳ vọng 2
s='abca', k=0 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Cắt lát chuỗi và cách "xoay" chuỗi

Trong Python, `s[a:b]` lấy các ký tự từ chỉ số `a` đến `b - 1`; `s[p:]` lấy từ `p` đến hết; `s[:p]` lấy `p` ký tự đầu (tiền tố độ dài `p`). Nếu chọn tiền tố `s[:p]` rồi chuyển xuống cuối, ta được `s[p:] + s[:p]`.

Ví dụ `s = "abcde"`, `p = 2`: tiền tố là `"ab"`, phần còn lại là `"cde"`, kết quả `"cdeab"`.

In [2]:
s = "abcde"
for p in range(len(s)):
    print(f"p={p}: tiền tố={s[:p]!r:8} phần còn lại={s[p:]!r:8} -> xoay = {s[p:] + s[:p]!r}")

p=0: tiền tố=''       phần còn lại='abcde'  -> xoay = 'abcde'
p=1: tiền tố='a'      phần còn lại='bcde'   -> xoay = 'bcdea'
p=2: tiền tố='ab'     phần còn lại='cde'    -> xoay = 'cdeab'
p=3: tiền tố='abc'    phần còn lại='de'     -> xoay = 'deabc'
p=4: tiền tố='abcd'   phần còn lại='e'      -> xoay = 'eabcd'


Khi `p = 0` tiền tố rỗng nên chuỗi không đổi. Khi `p = n` ta sẽ quay về đúng chuỗi ban đầu, vì vậy đề chỉ cho `p` chạy từ `0` đến `n - 1`: đủ `n` phép xoay khác nhau về **cách chọn**.

### 3.2. So sánh hai ký tự liền kề và ghép cặp bằng `zip`

Hai ký tự ở vị trí `i` và `i + 1` là `s[i]` và `s[i + 1]`. Để duyệt mọi cặp liền kề, ta có thể dùng `range(len(s) - 1)` (vì `i + 1` không được vượt chỉ số cuối) hoặc `zip(s, s[1:])` — hàm `zip` ghép phần tử cùng vị trí của hai dãy và dừng khi dãy ngắn hơn hết.

In [3]:
s = "aabba"
print("cách 1, dùng chỉ số:")
for i in range(len(s) - 1):
    print(f"  i={i}: {s[i]} và {s[i + 1]} -> {'bằng nhau' if s[i] == s[i + 1] else 'khác nhau'}")

print("cách 2, dùng zip(s, s[1:]):", list(zip(s, s[1:])))
print("cách 3, dùng pairwise(s):  ", list(pairwise(s)))

cách 1, dùng chỉ số:
  i=0: a và a -> bằng nhau
  i=1: a và b -> khác nhau
  i=2: b và b -> bằng nhau
  i=3: b và a -> khác nhau
cách 2, dùng zip(s, s[1:]): [('a', 'a'), ('a', 'b'), ('b', 'b'), ('b', 'a')]
cách 3, dùng pairwise(s):   [('a', 'a'), ('a', 'b'), ('b', 'b'), ('b', 'a')]


### 3.3. Cộng các giá trị `True`/`False`

Trong Python `True` bằng `1` và `False` bằng `0` khi cộng. Vì vậy `sum(a == b for a, b in zip(s, s[1:]))` đếm số cặp liền kề giống nhau chỉ trong một dòng.

In [4]:
print(True + True, True + False, False + False)      # 2 1 0

def score(t: str) -> int:
    """Điểm số của một chuỗi: số cặp liền kề giống nhau (không tính nối vòng)."""
    return sum(a == b for a, b in zip(t, t[1:]))

for t in ["aab", "aba", "baa", "aaaa", "abab"]:
    print(t, "->", score(t))

2 1 0
aab -> 1
aba -> 0
baa -> 1
aaaa -> 3
abab -> 0


### 3.4. Toán tử `%` và chỉ số "vòng tròn"

Với số nguyên không âm, `x % n` cho phần dư khi chia cho `n`, luôn nằm trong `0..n-1`. Ta dùng nó để **quay vòng chỉ số**: sau chỉ số `n - 1` là `0` (`(n - 1 + 1) % n = 0`), và trước chỉ số `0` là `n - 1` (`(0 - 1) % n = n - 1`).

Điểm khác với nhiều ngôn ngữ khác: trong Python `%` luôn trả số **cùng dấu với số chia**, nên `(-1) % 5 == 4` (không phải `-1`). Điều này làm cho `(p - 1) % n` an toàn cho mọi `p`.

In [5]:
n = 5
print([(i + 1) % n for i in range(n)])    # [1, 2, 3, 4, 0]  <- sau n-1 là 0
print([(i - 1) % n for i in range(n)])    # [4, 0, 1, 2, 3]  <- trước 0 là n-1
print((-1) % 5, (-6) % 5, 7 % 5)          # 4 4 2

[1, 2, 3, 4, 0]
[4, 0, 1, 2, 3]
4 4 2


### 3.5. Nối đôi chuỗi: `s + s` chứa mọi phép xoay

Mọi phép xoay của `s` đều là một **đoạn con độ dài `n`** của `s + s`, bắt đầu ở vị trí `p`. Ví dụ `s = "aab"` thì `s + s = "aabaab"`: các đoạn con độ dài 3 bắt đầu ở `0, 1, 2` là `"aab"`, `"aba"`, `"baa"` — đúng ba phép xoay của ví dụ 1.

In [6]:
s = "aab"
t = s + s
n = len(s)
print("s + s =", t)
for p in range(n):
    assert t[p:p + n] == s[p:] + s[:p]
    print(f"p={p}: t[{p}:{p + n}] = {t[p:p + n]!r}")

s + s = aabaab
p=0: t[0:3] = 'aab'
p=1: t[1:4] = 'aba'
p=2: t[2:5] = 'baa'


### 3.6. Tổng tiền tố (prefix sum) với `accumulate`

Nếu có dãy `a`, thì **tổng tiền tố** `pre[i] = a[0] + ... + a[i-1]`. Tổng đoạn `a[l..r-1]` bằng `pre[r] - pre[l]` trong `O(1)`. Thư viện chuẩn có `itertools.accumulate`: `list(accumulate(a, initial=0))` tạo `pre` với `pre[0] = 0`.

In [7]:
a = [1, 0, 1, 1, 0]
pre = list(accumulate(a, initial=0))
print("pre =", pre)                       # [0, 1, 1, 2, 3, 3]
print("tổng a[1..3] =", pre[4] - pre[1])  # 0 + 1 + 1 = 2

pre = [0, 1, 1, 2, 3, 3]
tổng a[1..3] = 2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy làm tay ví dụ `s = "aab"`: viết ra từng phép xoay và đếm số cặp giống nhau.

| `p` | tiền tố chuyển xuống cuối | chuỗi xoay | các cặp liền kề | điểm |
|---|---|---|---|---|
| 0 | `""` | `aab` | `aa` (bằng), `ab` (khác) | 1 |
| 1 | `"a"` | `aba` | `ab`, `ba` (đều khác) | 0 |
| 2 | `"aa"` | `baa` | `ba` (khác), `aa` (bằng) | 1 |

Điểm bằng `k = 1` ở `p = 0` và `p = 2` nên đáp án là `2`.

Cách nghĩ trực tiếp là đúng như vậy: **với mỗi `p` dựng chuỗi mới rồi đếm**. Ta sẽ viết code cho hành động này ở Cách 1, rồi quan sát xem các phép xoay "thay đổi" so với nhau ở đâu để tăng tốc.

In [8]:
s = "aab"
n = len(s)
print("p | chuỗi xoay | điểm")
for p in range(n):
    r = s[p:] + s[:p]
    print(f"{p} | {r:^10} | {sum(a == b for a, b in zip(r, r[1:]))}")

p | chuỗi xoay | điểm
0 |    aab     | 1
1 |    aba     | 0
2 |    baa     | 1


## 5. Cách 1 — Brute Force / baseline: dựng từng phép xoay

**Ý tưởng.** Với mỗi `p` từ `0` đến `n - 1`:

1. dựng `r = s[p:] + s[:p]`;
2. đếm số vị trí `i` trong `0..n-2` có `r[i] == r[i + 1]`;
3. nếu điểm bằng `k` thì tăng biến đếm.

Code dưới đây giữ đúng ba bước đó, không tối ưu gì cả.

In [9]:
def count_rotations_bruteforce(s: str, k: int) -> int:
    n = len(s)
    answer = 0
    for p in range(n):                      # độ dài tiền tố chuyển xuống cuối
        r = s[p:] + s[:p]                   # dựng chuỗi xoay: O(n)
        pairs = 0
        for i in range(n - 1):              # đếm cặp liền kề giống nhau: O(n)
            if r[i] == r[i + 1]:
                pairs += 1
        if pairs == k:
            answer += 1
    return answer


for s, k, expected in examples:
    print(f"s={s!r}, k={k} -> {count_rotations_bruteforce(s, k)} (kỳ vọng {expected})")

s='aab', k=1 -> 2 (kỳ vọng 2)
s='abca', k=0 -> 1 (kỳ vọng 1)


**Độ phức tạp (Cách 1).** Gọi `n` là độ dài chuỗi. Có `n` phép xoay; mỗi phép dựng chuỗi mất `O(n)` và đếm cặp mất `O(n)`. Tổng **Time `O(n²)`**; chuỗi tạm `r` cần **Space `O(n)`**.

**Nút thắt.** Với `n ≤ 100` cách này vẫn nộp được, nhưng ta đang làm lại gần như cùng một công việc `n` lần: hai phép xoay liên tiếp `"aab"` và `"aba"` chia sẻ gần hết các cặp liền kề, chỉ khác ở chỗ "khâu" đầu–cuối. Nếu giữ lại phần dùng chung, ta chỉ cần xử lý phần khác nhau.

## 6. Ý tưởng thuật toán / bước đột phá

**Hãy xếp các ký tự của `s` quanh một vòng tròn.** Trên vòng tròn có đúng `n` cặp liền kề: `(0,1), (1,2), ..., (n-2,n-1)` và cặp nối đuôi–đầu `(n-1, 0)`. Gọi

- `cyc[i] = 1` nếu `s[i] == s[(i + 1) % n]`, ngược lại `0` (cặp thứ `i` trên vòng),
- `T = sum(cyc)` là tổng số cặp giống nhau **trên vòng tròn**.

**Một phép xoay chính là "cắt vòng tròn ra thành đoạn thẳng" tại một chỗ.** Phép xoay bắt đầu ở `p` đọc các ký tự `s[p], s[p+1], ..., s[p-1]` (theo vòng). Các cặp liền kề của chuỗi thẳng này là mọi cặp trên vòng, **trừ đúng một cặp**: cặp `(p-1, p)` bị cắt đứt (hai ký tự này nằm ở hai đầu chuỗi thẳng). Vậy:

```
điểm(p) = T − cyc[(p − 1) mod n]
```

Điểm chỉ nhận **hai giá trị**: `T` (khi cặp bị cắt là cặp khác nhau) hoặc `T − 1` (khi cặp bị cắt là cặp bằng nhau). Công thức này cho phép:

- tính điểm mọi phép xoay trong `O(n)` tổng cộng (Cách 2);
- hoặc cập nhật điểm từ phép xoay này sang phép xoay kế tiếp trong `O(1)` (Cách 3);
- hoặc đếm thẳng theo `T` mà không cần duyệt từng `p` (Cách 5).

In [10]:
s = "aab"
n = len(s)
cyc = [int(s[i] == s[(i + 1) % n]) for i in range(n)]
T = sum(cyc)
print("cyc =", cyc, " T =", T)
for p in range(n):
    r = s[p:] + s[:p]
    brute = sum(a == b for a, b in zip(r, r[1:]))
    formula = T - cyc[(p - 1) % n]
    print(f"p={p}: xoay={r!r}, điểm brute={brute}, điểm công thức={formula}")
    assert brute == formula

cyc = [1, 0, 0]  T = 1
p=0: xoay='aab', điểm brute=1, điểm công thức=1
p=1: xoay='aba', điểm brute=0, điểm công thức=0
p=2: xoay='baa', điểm brute=1, điểm công thức=1


## 7. Các cách tiếp theo

### Cách 2 — Công thức vòng tròn: `điểm(p) = T − cyc[(p − 1) mod n]`

**Giải thích.**

1. Tính mảng `cyc` có `n` phần tử (so sánh `s[i]` với `s[(i + 1) % n]`, dùng `%` để quay vòng).
2. Tính `T = sum(cyc)`.
3. Với mỗi `p`, điểm là `T - cyc[(p - 1) % n]`; đếm các `p` có điểm bằng `k`.

In [11]:
def count_rotations_cyclic(s: str, k: int) -> int:
    n = len(s)
    cyc = [1 if s[i] == s[(i + 1) % n] else 0 for i in range(n)]   # n cặp trên vòng tròn
    total = sum(cyc)
    answer = 0
    for p in range(n):
        if total - cyc[(p - 1) % n] == k:        # trừ cặp bị cắt đứt ở chỗ khâu
            answer += 1
    return answer


for s, k, expected in examples:
    print(f"s={s!r}, k={k} -> {count_rotations_cyclic(s, k)} (kỳ vọng {expected})")

s='aab', k=1 -> 2 (kỳ vọng 2)
s='abca', k=0 -> 1 (kỳ vọng 1)


In [12]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def countRotations(self, s: str, k: int) -> int:
        n = len(s)
        cyc = [1 if s[i] == s[(i + 1) % n] else 0 for i in range(n)]
        total = sum(cyc)
        return sum(1 for p in range(n) if total - cyc[(p - 1) % n] == k)

**Độ phức tạp Cách 2:** **Time `O(n)`** (tính `cyc` một lượt, rồi duyệt `p` một lượt), **Space `O(n)`** cho mảng `cyc`.

### Cách 3 — Cập nhật trượt `O(1)` mỗi lần xoay (không cần mảng phụ)

**Giải thích.** Khi đi từ phép xoay `p` sang `p + 1`, ký tự `s[p]` rời đầu chuỗi xuống cuối:

- **mất** cặp `(s[p], s[p+1])` — cặp đứng đầu chuỗi cũ;
- **thêm** cặp `(s[p-1], s[p])` — cặp mới ở cuối chuỗi (ký tự cuối cũ `s[p-1]` nay đứng cạnh `s[p]`).

Vậy `điểm(p+1) = điểm(p) − [s[p]==s[p+1]] + [s[p-1]==s[p]]`, mọi chỉ số lấy `% n`. Chỉ cần tính điểm gốc (`p = 0`) bằng một lượt đếm rồi cập nhật `n - 1` lần.

In [13]:
def count_rotations_sliding(s: str, k: int) -> int:
    n = len(s)
    score = sum(a == b for a, b in zip(s, s[1:]))     # điểm của phép xoay p = 0
    answer = 1 if score == k else 0
    for p in range(n - 1):                            # chuyển từ p sang p + 1
        score -= s[p] == s[(p + 1) % n]               # mất cặp ở đầu: (s[p], s[p+1])
        score += s[(p - 1) % n] == s[p]               # thêm cặp ở cuối: (s[p-1], s[p])
        if score == k:
            answer += 1
    return answer


for s, k, expected in examples:
    print(f"s={s!r}, k={k} -> {count_rotations_sliding(s, k)} (kỳ vọng {expected})")

s='aab', k=1 -> 2 (kỳ vọng 2)
s='abca', k=0 -> 1 (kỳ vọng 1)


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`** (chỉ vài biến). Đây cũng là hướng của lời giải tham khảo trong editorial.

### Cách 4 — Tổng tiền tố trên `s + s`

**Giải thích.** Gọi `t = s + s` và `eq[i] = 1` nếu `t[i] == t[i + 1]` (với `i = 0..2n-2`). Phép xoay `p` chính là `t[p : p + n]`, có các cặp liền kề `eq[p], eq[p+1], ..., eq[p+n-2]` — tổng của một **cửa sổ** `n - 1` phần tử. Với tổng tiền tố `pre`, điểm là `pre[p + n - 1] - pre[p]`.

In [14]:
def count_rotations_prefix(s: str, k: int) -> int:
    n = len(s)
    t = s + s
    eq = [1 if a == b else 0 for a, b in zip(t, t[1:])]      # độ dài 2n - 1
    pre = list(accumulate(eq, initial=0))                     # pre[i] = eq[0] + ... + eq[i-1]
    answer = 0
    for p in range(n):
        window = pre[p + n - 1] - pre[p]                      # tổng eq[p .. p+n-2]
        if window == k:
            answer += 1
    return answer


for s, k, expected in examples:
    print(f"s={s!r}, k={k} -> {count_rotations_prefix(s, k)} (kỳ vọng {expected})")

s='aab', k=1 -> 2 (kỳ vọng 2)
s='abca', k=0 -> 1 (kỳ vọng 1)


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** (chuỗi `s + s`, mảng `eq` và `pre`). Ưu điểm: không phải suy luận về "cặp bị cắt", chỉ cần tin rằng mọi phép xoay là một cửa sổ trên `s + s`.

### Cách 5 — Công thức đếm đóng (không cần duyệt từng `p`)

**Giải thích.** Từ Cách 2, điểm của mỗi phép xoay chỉ là `T` hoặc `T − 1`:

- điểm bằng `T` khi `cyc[(p-1) % n] = 0`: có `n − T` giá trị `p` như vậy;
- điểm bằng `T − 1` khi `cyc[(p-1) % n] = 1`: có `T` giá trị `p` như vậy.

Do đó: nếu `k == T` đáp án là `n − T`; nếu `k == T − 1` đáp án là `T`; còn lại là `0`. (Hai trường hợp không thể cùng xảy ra vì `T ≠ T − 1`.)

In [15]:
def count_rotations_closed(s: str, k: int) -> int:
    n = len(s)
    total = sum(s[i] == s[(i + 1) % n] for i in range(n))     # T
    if k == total:
        return n - total
    if k == total - 1:
        return total
    return 0


for s, k, expected in examples:
    print(f"s={s!r}, k={k} -> {count_rotations_closed(s, k)} (kỳ vọng {expected})")

s='aab', k=1 -> 2 (kỳ vọng 2)
s='abca', k=0 -> 1 (kỳ vọng 1)


**Độ phức tạp Cách 5:** **Time `O(n)`** (chỉ để tính `T`), **Space `O(1)`**.

> Cách 2 và Cách 5 dựa trên cùng một ý; Cách 3 là dạng "cập nhật dần". Khi nộp bài, bất kỳ cách nào trong ba cách này đều ổn; Cách 2 dễ giải thích nhất, Cách 5 ngắn nhất.

## 8. TRACE từng bước

Ta theo dõi `s = "abca"` (ví dụ 2, `k = 0`) bằng Cách 2 và Cách 3, rồi một ví dụ có nhiều chữ giống nhau liền nhau `s = "aabaa"`.

In [16]:
def trace_cyclic(s: str, k: int):
    n = len(s)
    cyc = [1 if s[i] == s[(i + 1) % n] else 0 for i in range(n)]
    total = sum(cyc)
    print(f"s={s!r}, k={k}, n={n}")
    print("cặp trên vòng:", [f"{s[i]}{s[(i + 1) % n]}" for i in range(n)], "-> cyc =", cyc, "-> T =", total)
    answer = 0
    for p in range(n):
        cut = (p - 1) % n
        score = total - cyc[cut]
        r = s[p:] + s[:p]
        hit = score == k
        print(f"  p={p}: xoay={r!r}, cặp bị cắt cyc[{cut}]={cyc[cut]}, điểm = {total} - {cyc[cut]} = {score}"
              f" {'<- đúng k' if hit else ''}")
        answer += hit
    print("  => đáp án:", answer)
    return answer


assert trace_cyclic("abca", 0) == 1
print()
assert trace_cyclic("aab", 1) == 2

s='abca', k=0, n=4
cặp trên vòng: ['ab', 'bc', 'ca', 'aa'] -> cyc = [0, 0, 0, 1] -> T = 1
  p=0: xoay='abca', cặp bị cắt cyc[3]=1, điểm = 1 - 1 = 0 <- đúng k
  p=1: xoay='bcaa', cặp bị cắt cyc[0]=0, điểm = 1 - 0 = 1 
  p=2: xoay='caab', cặp bị cắt cyc[1]=0, điểm = 1 - 0 = 1 
  p=3: xoay='aabc', cặp bị cắt cyc[2]=0, điểm = 1 - 0 = 1 
  => đáp án: 1

s='aab', k=1, n=3
cặp trên vòng: ['aa', 'ab', 'ba'] -> cyc = [1, 0, 0] -> T = 1
  p=0: xoay='aab', cặp bị cắt cyc[2]=0, điểm = 1 - 0 = 1 <- đúng k
  p=1: xoay='aba', cặp bị cắt cyc[0]=1, điểm = 1 - 1 = 0 
  p=2: xoay='baa', cặp bị cắt cyc[1]=0, điểm = 1 - 0 = 1 <- đúng k
  => đáp án: 2


In [17]:
def trace_sliding(s: str, k: int):
    n = len(s)
    score = sum(a == b for a, b in zip(s, s[1:]))
    answer = 1 if score == k else 0
    print(f"s={s!r}, k={k}; p=0: xoay={s!r}, điểm gốc={score} {'<- đúng k' if score == k else ''}")
    for p in range(n - 1):
        lost = s[p] == s[(p + 1) % n]
        gained = s[(p - 1) % n] == s[p]
        score = score - lost + gained
        r = s[p + 1:] + s[:p + 1]
        print(f"  p={p + 1}: xoay={r!r}, mất cặp ({s[p]},{s[(p + 1) % n]})={int(lost)}, "
              f"thêm cặp ({s[(p - 1) % n]},{s[p]})={int(gained)}, điểm={score} {'<- đúng k' if score == k else ''}")
        answer += score == k
    print("  => đáp án:", answer)
    return answer


assert trace_sliding("aabaa", 2) == count_rotations_bruteforce("aabaa", 2)

s='aabaa', k=2; p=0: xoay='aabaa', điểm gốc=2 <- đúng k
  p=1: xoay='abaaa', mất cặp (a,a)=1, thêm cặp (a,a)=1, điểm=2 <- đúng k
  p=2: xoay='baaaa', mất cặp (a,b)=0, thêm cặp (a,a)=1, điểm=3 
  p=3: xoay='aaaab', mất cặp (b,a)=0, thêm cặp (a,b)=0, điểm=3 
  p=4: xoay='aaaba', mất cặp (a,a)=1, thêm cặp (b,a)=0, điểm=2 <- đúng k
  => đáp án: 3


Bảng tóm tắt (Cách 2, `s = "abca"`, `k = 0`):

| `p` | chuỗi xoay | cặp bị cắt `(p-1, p)` | `cyc[(p-1)%n]` | điểm `= T − cyc` | có bằng `k = 0`? |
|---|---|---|---|---|---|
| 0 | `abca` | `(3, 0)` = `aa` | 1 | 0 | có |
| 1 | `bcaa` | `(0, 1)` = `ab` | 0 | 1 | không |
| 2 | `caab` | `(1, 2)` = `bc` | 0 | 1 | không |
| 3 | `aabc` | `(2, 3)` = `ca` | 0 | 1 | không |

Ở đây `T = 1` (chỉ có cặp nối `aa` giữa cuối và đầu). Phép xoay duy nhất có điểm `0` là phép xoay mà chỗ cắt đúng vào cặp `aa` đó (`p = 0`), đúng như ví dụ 2.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (công thức vòng tròn): ngắn, dễ giải thích và `O(n)`. Đúng class `Solution`, đúng method `countRotations(self, s: str, k: int) -> int`.

In [18]:
class Solution:
    def countRotations(self, s: str, k: int) -> int:
        n = len(s)
        # cyc[i] = 1 nếu s[i] và s[i + 1] (quay vòng) giống nhau
        cyc = [1 if s[i] == s[(i + 1) % n] else 0 for i in range(n)]
        total = sum(cyc)
        answer = 0
        for p in range(n):                          # p = độ dài tiền tố chuyển xuống cuối
            if total - cyc[(p - 1) % n] == k:       # phép xoay p làm mất cặp (p - 1, p)
                answer += 1
        return answer


for s, k, expected in examples:
    assert Solution().countRotations(s, k) == expected
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Mọi phép xoay là một vòng tròn bị cắt:** viết `s` lên vòng tròn. Phép xoay `p` là đọc vòng tròn bắt đầu từ ký tự `p`, đi hết một vòng `n` ký tự. Các cặp liền kề của chuỗi thẳng là các cặp liền kề trên vòng tròn, trừ cặp ở "chỗ cắt" — cặp `(p-1, p)` — vì hai ký tự này nằm ở hai đầu chuỗi thẳng và không đứng cạnh nhau.
- **Đúng `n - 1` cặp còn lại:** vòng tròn có `n` cặp, bỏ 1 cặp còn `n - 1` cặp, khớp với số chỉ số `i` trong `0..n-2` của đề.
- **Công thức điểm:** điểm = (số cặp bằng nhau trên vòng) − (cặp bị cắt có bằng nhau không) = `T − cyc[(p-1) mod n]`.
- **Mỗi `p` được đếm đúng một lần:** vòng lặp `p = 0..n-1` ứng với đúng các độ dài tiền tố `0..n-1` của đề.
- **Trường hợp `n = 2`:** vòng tròn có 2 cặp `(0,1)` và `(1,0)` là cùng một cặp ký tự; `T` là `0` hoặc `2`; mỗi phép xoay có điểm `T - cyc = T / 2`... nghĩa là `0` hoặc `1`. Điều này khớp với việc chuỗi 2 ký tự có đúng 1 cặp.
- **Cách 3 (trượt):** đã suy ra từ công thức, vì `điểm(p+1) - điểm(p) = cyc[p-1] - cyc[p]`; `cyc[p]` là cặp `(s[p], s[p+1])` và `cyc[p-1]` là cặp `(s[p-1], s[p])`.

## 11. Phân tích Time / Space Complexity

Gọi `n` là độ dài của `s` (`n ≤ 100` theo đề).

- **Cách 1 (dựng từng xoay):** Time `O(n²)`, Space `O(n)`.
- **Cách 2 (công thức vòng tròn):** Time `O(n)`, Space `O(n)`.
- **Cách 3 (cập nhật trượt):** Time `O(n)`, Space `O(1)`.
- **Cách 4 (tổng tiền tố trên `s + s`):** Time `O(n)`, Space `O(n)`.
- **Cách 5 (công thức đóng):** Time `O(n)`, Space `O(1)`.
- Với mọi cách, trường hợp trung bình và xấu nhất như nhau, vì ta luôn duyệt hết chuỗi. Không thể nhanh hơn `O(n)` vì phải đọc hết `s` ít nhất một lần.

## 12. Bộ test

Ta gom các cách vào từ điển `impls`, thêm một **lời giải độc lập** `oracle` (dựng chuỗi xoay bằng chỉ số vòng, đếm bằng vòng lặp riêng, không dùng bất kỳ kỹ thuật nào ở trên) để đối chiếu.

In [19]:
sol = Solution()
impls = {
    "bruteforce": count_rotations_bruteforce,
    "cyclic": count_rotations_cyclic,
    "sliding": count_rotations_sliding,
    "prefix": count_rotations_prefix,
    "closed": count_rotations_closed,
    "submit": sol.countRotations,
}


def oracle(s: str, k: int) -> int:
    """Đáp án độc lập: dựng chuỗi xoay bằng chỉ số (p + t) % n, đếm bằng vòng lặp trần."""
    n = len(s)
    answer = 0
    for p in range(n):
        rotated = [s[(p + t) % n] for t in range(n)]
        pairs = 0
        for i in range(n - 1):
            if rotated[i] == rotated[i + 1]:
                pairs += 1
        if pairs == k:
            answer += 1
    return answer


for s, k, expected in examples:
    assert oracle(s, k) == expected
    for name, fn in impls.items():
        assert fn(s, k) == expected, (name, s, k, fn(s, k), expected)
print("PASS ví dụ của đề (mọi cách và oracle độc lập)")

PASS ví dụ của đề (mọi cách và oracle độc lập)


### 12.1. Các trường hợp biên

In [20]:
def check_all(s, k, expected, label):
    assert oracle(s, k) == expected, (label, "oracle", oracle(s, k), expected)
    for name, fn in impls.items():
        got = fn(s, k)
        assert got == expected, (label, name, got, expected)
    print("PASS", label, "->", expected)


check_all("aa", 1, 2, "n = 2, hai ký tự giống nhau: cả 2 phép xoay đều điểm 1")
check_all("aa", 0, 0, "n = 2, giống nhau nhưng k = 0")
check_all("ab", 0, 2, "n = 2, khác nhau: mọi phép xoay điểm 0")
check_all("ab", 1, 0, "n = 2, khác nhau nhưng k = 1")
check_all("abc", 0, 3, "toàn ký tự khác nhau: mọi phép xoay điểm 0")
check_all("aaaa", 3, 4, "toàn ký tự giống nhau: mọi phép xoay điểm n - 1")
check_all("aaaa", 2, 0, "toàn giống nhau, k = n - 2")
check_all("abab", 0, 4, "xen kẽ: không có cặp giống nhau")
check_all("aabb", 1, 2, "aabb: T = 2 nên điểm 1 xảy ra khi cắt vào aa hoặc bb")
check_all("a" * 100, 99, 100, "chuỗi dài nhất của đề, toàn giống nhau")
check_all("ab" * 50, 0, 100, "chuỗi dài nhất, xen kẽ")

PASS n = 2, hai ký tự giống nhau: cả 2 phép xoay đều điểm 1 -> 2
PASS n = 2, giống nhau nhưng k = 0 -> 0
PASS n = 2, khác nhau: mọi phép xoay điểm 0 -> 2
PASS n = 2, khác nhau nhưng k = 1 -> 0
PASS toàn ký tự khác nhau: mọi phép xoay điểm 0 -> 3
PASS toàn ký tự giống nhau: mọi phép xoay điểm n - 1 -> 4
PASS toàn giống nhau, k = n - 2 -> 0
PASS xen kẽ: không có cặp giống nhau -> 4
PASS aabb: T = 2 nên điểm 1 xảy ra khi cắt vào aa hoặc bb -> 2
PASS chuỗi dài nhất của đề, toàn giống nhau -> 100
PASS chuỗi dài nhất, xen kẽ -> 100


### 12.2. Tính chất và phản ví dụ

In [21]:
# Tính chất 1: mọi phép xoay có điểm thuộc {T, T-1}; tổng các điểm = n*T - T = (n-1)*T
random.seed(4043)
for _ in range(300):
    n = random.randint(2, 12)
    s = "".join(random.choice("ab") for _ in range(n))
    cyc = [s[i] == s[(i + 1) % n] for i in range(n)]
    T = sum(cyc)
    scores = []
    for p in range(n):
        r = s[p:] + s[:p]
        scores.append(sum(a == b for a, b in zip(r, r[1:])))
    assert set(scores) <= {T, T - 1}, (s, scores, T)
    assert sum(scores) == (n - 1) * T, (s, scores, T)
print("PASS tính chất: điểm luôn thuộc {T, T-1} và tổng điểm = (n-1)*T")

# Tính chất 2: tổng đáp án với mọi k = n (mỗi phép xoay có đúng một điểm)
for _ in range(200):
    n = random.randint(2, 12)
    s = "".join(random.choice("abc") for _ in range(n))
    assert sum(sol.countRotations(s, k) for k in range(n)) <= n
    assert sum(sol.countRotations(s, k) for k in range(n + 1)) == n
print("PASS tính chất: các k khác nhau chia đều n phép xoay")

# Phản ví dụ cho cách gộp các phép xoay giống nhau (dùng set): "aa" có hai phép xoay đều là "aa"
distinct = {("aa"[p:] + "aa"[:p]) for p in range(2)}
print("số chuỗi xoay khác nhau của 'aa':", len(distinct), "nhưng đáp án đúng cho k = 1 là", sol.countRotations("aa", 1))
assert len(distinct) == 1 and sol.countRotations("aa", 1) == 2

PASS tính chất: điểm luôn thuộc {T, T-1} và tổng điểm = (n-1)*T
PASS tính chất: các k khác nhau chia đều n phép xoay
số chuỗi xoay khác nhau của 'aa': 1 nhưng đáp án đúng cho k = 1 là 2


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [22]:
random.seed(2026)
for trial in range(3000):
    n = random.randint(2, 14)
    alphabet = random.choice(["ab", "abc", "abcdefghij", "a"])
    s = "".join(random.choice(alphabet) for _ in range(n))
    k = random.randint(0, n - 1)
    expected = oracle(s, k)
    for name, fn in impls.items():
        assert fn(s, k) == expected, (name, s, k, fn(s, k), expected)

for trial in range(500):                      # chuỗi dài tới 100
    n = random.randint(50, 100)
    s = "".join(random.choice("ab") for _ in range(n))
    k = random.randint(0, n - 1)
    expected = oracle(s, k)
    for name, fn in impls.items():
        assert fn(s, k) == expected, (name, s, k)
print("PASS 3500 input ngẫu nhiên: mọi cách khớp oracle độc lập")

PASS 3500 input ngẫu nhiên: mọi cách khớp oracle độc lập


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ đếm cặp trong chuỗi gốc, quên rằng phép xoay đổi chỗ "khâu"

In [23]:
def sai_chi_dem_goc(s, k):
    # tưởng mọi phép xoay có cùng điểm với chuỗi gốc
    score = sum(a == b for a, b in zip(s, s[1:]))
    return len(s) if score == k else 0


print("sai:", sai_chi_dem_goc("aab", 1), "| đúng:", Solution().countRotations("aab", 1))

sai: 3 | đúng: 2


**Vì sao sai:** điểm thay đổi khi xoay: `"aab"` có điểm 1 nhưng `"aba"` có điểm 0. Ở mỗi lần xoay, một cặp liền kề biến mất và một cặp mới xuất hiện. Cần đếm lại (hoặc cập nhật) cho từng phép xoay. Code đúng là Cách 1 đến Cách 5 ở trên.

### Lỗi 2: Quên trừ đi cặp bị cắt (cho rằng mọi phép xoay có điểm `T`)

In [24]:
def sai_quen_cap_bi_cat(s, k):
    # đếm cặp liền kề trên vòng tròn rồi cho rằng mọi phép xoay đều có điểm T
    n = len(s)
    total = sum(s[i] == s[(i + 1) % n] for i in range(n))
    return n if total == k else 0


print("sai:", sai_quen_cap_bi_cat("aab", 1), "| đúng:", Solution().countRotations("aab", 1))
print("sai:", sai_quen_cap_bi_cat("abca", 0), "| đúng:", Solution().countRotations("abca", 0))

sai: 3 | đúng: 2
sai: 0 | đúng: 1


**Vì sao sai:** `T` là số cặp trên **vòng tròn** (`n` cặp), còn chuỗi thẳng chỉ có `n - 1` cặp. Mỗi phép xoay phải trừ cặp `(p-1, p)` bị cắt đứt: `điểm = T − cyc[(p−1) % n]`. Với `"aab"`, `T = 1` (chỉ có cặp `(0,1)` = `aa`), nhưng phép xoay `"aba"` (`p = 1`) đã bị cắt đúng vào cặp `aa` đó nên chỉ có điểm 0. Vậy không phải phép xoay nào cũng có điểm `T`.

### Lỗi 3: Gộp các phép xoay trùng chuỗi (dùng `set`)

In [25]:
def sai_dung_set(s, k):
    n = len(s)
    rotations = {s[p:] + s[:p] for p in range(n)}       # gộp các phép xoay trùng nhau
    return sum(sum(a == b for a, b in zip(r, r[1:])) == k for r in rotations)


print("sai:", sai_dung_set("aa", 1), "| đúng:", Solution().countRotations("aa", 1))
print("sai:", sai_dung_set("abab", 0), "| đúng:", Solution().countRotations("abab", 0))

sai: 1 | đúng: 2
sai: 2 | đúng: 4


**Vì sao sai:** đề đếm **số phép xoay** (mỗi độ dài tiền tố `0..n-1` là một phép), không phải số chuỗi khác nhau. `"aa"` có hai phép xoay cho cùng chuỗi `"aa"`, đáp án đúng là `2`, không phải `1`. Với `"abab"` hai phép xoay `p = 0, 2` cho `"abab"` và `p = 1, 3` cho `"baba"`; đáp án phải là `4`.

### Lỗi 4: Bỏ mất cặp nối vòng khi dựng `cyc` (dùng `range(n - 1)`)

In [26]:
def sai_bo_cap_noi_vong(s, k):
    n = len(s)
    cyc = [1 if s[i] == s[i + 1] else 0 for i in range(n - 1)]     # thiếu cặp (n-1, 0)
    total = sum(cyc)
    return sum(1 for p in range(n) if total - cyc[(p - 1) % (n - 1)] == k)


print("sai:", sai_bo_cap_noi_vong("abca", 0), "| đúng:", Solution().countRotations("abca", 0))
print("sai:", sai_bo_cap_noi_vong("aabaa", 1), "| đúng:", Solution().countRotations("aabaa", 1))

sai: 4 | đúng: 1
sai: 3 | đúng: 0


**Vì sao sai:** trên vòng tròn có **`n` cặp**, gồm cả cặp đuôi–đầu `(n-1, 0)`. Nếu chỉ lấy `n - 1` cặp của chuỗi gốc, `T` bị thiếu và chỉ số `% (n - 1)` làm lệch đối tượng bị cắt. Ví dụ `"abca"` có cặp `aa` nối vòng quan trọng nhất của bài: bỏ nó làm đáp án thành `4` thay vì `1`. Chú ý: nếu cắt nhầm sang `cyc[p+1]` thì **số lượng** phép xoay đếm được vẫn đúng (vì mọi chỉ số vẫn được duyệt đủ một lần); lỗi thật nằm ở việc **thiếu cặp** hoặc **sai `T`**.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**xoay vòng / cyclic shift / rotation**" của chuỗi hoặc mảng: nghĩ ngay tới **vòng tròn** hoặc **nối đôi `s + s`** (mọi phép xoay là một cửa sổ độ dài `n`).
- Đề hỏi "một thống kê trên **mọi** phép xoay" và `n` nhỏ: brute force `O(n²)` luôn là bước đầu tốt; sau đó tìm phần **thay đổi giữa hai phép xoay liên tiếp** để cập nhật `O(1)`.
- "Cặp liền kề" trên chuỗi thẳng = cặp trên vòng tròn **trừ cặp bị cắt**: cách nhìn này đổi bài toán thành tính trên mảng vòng cố định.
- Các giá trị điểm chỉ nhận ít giá trị (ở đây `T` hoặc `T-1`) → có thể đếm bằng công thức thay vì duyệt.
- Dùng `(i + 1) % n` và `(i - 1) % n` để quay vòng chỉ số; Python xử lý số âm đúng nên `(p - 1) % n` luôn hợp lệ.
- Luôn tự viết một oracle brute force riêng để đối chiếu trên input ngẫu nhiên nhỏ: các bài vòng tròn rất dễ lệch một đơn vị.

## 15. Biến thể của bài

1. **Đếm cả cặp nối đầu–cuối:** nếu "điểm" tính cả cặp `(n-1, 0)` của chuỗi xoay thì mọi phép xoay có điểm `T`: câu trả lời là `n` nếu `k == T`, ngược lại `0`.
2. **Điểm bằng "ít nhất `k`"** hoặc **"nhiều nhất `k`"**: đếm theo công thức đóng với hai giá trị `T` và `T-1`.
3. **Đếm số chuỗi xoay khác nhau** có điểm `k` (dùng `set`; nhưng khác với đề này).
4. **Xoay mảng số nguyên** và đếm các cặp liền kề thỏa một quan hệ khác (hơn kém đúng 1, cùng chẵn lẻ...): cùng khung vòng tròn, chỉ đổi biểu thức so sánh.
5. **Tìm phép xoay có điểm lớn nhất/nhỏ nhất** thay vì đếm.

In [27]:
def count_rotations_general(s: str, k: int, same=lambda a, b: a == b) -> int:
    """Biến thể 4: quan hệ 'cặp tốt' tùy ý, vẫn dùng công thức vòng tròn."""
    n = len(s)
    cyc = [1 if same(s[i], s[(i + 1) % n]) else 0 for i in range(n)]
    total = sum(cyc)
    return sum(1 for p in range(n) if total - cyc[(p - 1) % n] == k)


def count_rotations_at_least(s: str, k: int) -> int:
    """Biến thể 2: điểm >= k."""
    n = len(s)
    total = sum(s[i] == s[(i + 1) % n] for i in range(n))
    return (n - total if total >= k else 0) + (total if total - 1 >= k else 0)


assert count_rotations_general("aab", 1) == 2
for trial in range(200):
    s = "".join(random.choice("ab") for _ in range(random.randint(2, 10)))
    k = random.randint(0, len(s) - 1)
    brute = sum(1 for p in range(len(s)) if sum(a == b for a, b in zip(s[p:] + s[:p], (s[p:] + s[:p])[1:])) >= k)
    assert count_rotations_at_least(s, k) == brute, (s, k)
print("PASS biến thể (quan hệ tùy ý, và điểm >= k)")

PASS biến thể (quan hệ tùy ý, và điểm >= k)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Dựng từng phép xoay | `s[p:] + s[:p]`, đếm cặp | `O(n²)` | `O(n)` | hiểu đề; đối chiếu |
| 2. Công thức vòng tròn | `T − cyc[(p−1)%n]` | `O(n)` | `O(n)` | **cách nộp bài**, dễ giải thích |
| 3. Cập nhật trượt | mất 1 cặp, thêm 1 cặp mỗi lần xoay | `O(n)` | `O(1)` | khi cần tiết kiệm bộ nhớ |
| 4. Tổng tiền tố trên `s + s` | điểm = tổng cửa sổ `n-1` phần tử | `O(n)` | `O(n)` | quan hệ cặp phức tạp hơn |
| 5. Công thức đóng | điểm chỉ `T` hoặc `T-1`: đếm | `O(n)` | `O(1)` | ngắn nhất, cần hiểu rõ ý Cách 2 |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"hello"[2:] + "hello"[:2]`, `(-3) % 7`, `sum([True, False, True])`, `Solution().countRotations("ab", 0)`, `Solution().countRotations("aaa", 2)`.

**Bài 2 (tính tay).** Với `s = "aabaa"`: tính mảng `cyc`, `T`, rồi điểm của từng phép xoay `p = 0..4`. Với `k = 2` và `k = 1` đáp án là bao nhiêu?

**Bài 3 (điền TODO).** Hoàn thành `rotations_todo(s, k)` theo Cách 2.

In [28]:
def rotations_todo(s, k):
    n = len(s)
    # TODO 1: tạo cyc = [1 nếu s[i] == s[(i + 1) % n] ngược lại 0 for i in range(n)]
    # TODO 2: total = tổng các phần tử của cyc
    # TODO 3: đếm các p trong range(n) mà total - cyc[(p - 1) % n] == k
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `rotations_sliding2(s, k)` theo Cách 3 (cập nhật trượt) mà không nhìn code mẫu. Kiểm tra bằng hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `best_rotation(s)` trả về **chỉ số `p`** của phép xoay có điểm **lớn nhất** (nếu có nhiều, trả về `p` nhỏ nhất). (b) Viết `count_distinct_rotations(s, k)` đếm số **chuỗi xoay khác nhau** (không phải số phép xoay) có điểm `k`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"hello"[2:]` là `"llo"`. `%` trong Python với số chia dương luôn cho kết quả không âm. `True` là `1`. Bài 2: viết `aabaa` quanh vòng tròn và so sánh từng cặp, kể cả cặp cuối–đầu.
- **Mức 2 (vừa):** Bài 1: kết quả lần lượt `"llohe"`, `4`, `2`, `2`, `3`. Bài 2: các cặp `aa, ab, ba, aa, aa` nên `cyc = [1,0,0,1,1]`, `T = 3`. Bài 3: ba TODO trong comment. Bài 5(a): điểm lớn nhất là `T` (khi cặp bị cắt là cặp khác nhau), nếu mọi cặp đều bằng nhau thì là `T - 1`.
- **Mức 3 (sâu):** Bài 2: điểm các phép xoay là `T - cyc[(p-1)%5]`: `p=0: 3-1=2`, `p=1: 3-1=2`, `p=2: 3-0=3`, `p=3: 3-0=3`, `p=4: 3-1=2`. Vậy `k=2` cho `3`, `k=1` cho `0`. Bài 5(a): `max(range(n), key=lambda p: (score(p), -p))`. Bài 5(b): dựng set các chuỗi xoay rồi chấm điểm từng chuỗi.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert "hello"[2:] + "hello"[:2] == "llohe"
assert (-3) % 7 == 4
assert sum([True, False, True]) == 2
assert Solution().countRotations("ab", 0) == 2
assert Solution().countRotations("aaa", 2) == 3

# Bài 2
s2 = "aabaa"
cyc2 = [int(s2[i] == s2[(i + 1) % 5]) for i in range(5)]
assert cyc2 == [1, 0, 0, 1, 1] and sum(cyc2) == 3
assert [3 - cyc2[(p - 1) % 5] for p in range(5)] == [2, 2, 3, 3, 2]
assert Solution().countRotations(s2, 2) == 3
assert Solution().countRotations(s2, 1) == 0
assert Solution().countRotations(s2, 3) == 2

# Bài 3
def rotations_todo_ans(s, k):
    n = len(s)
    cyc = [1 if s[i] == s[(i + 1) % n] else 0 for i in range(n)]
    total = sum(cyc)
    return sum(1 for p in range(n) if total - cyc[(p - 1) % n] == k)

for s, k, expected in examples:
    assert rotations_todo_ans(s, k) == expected

# Bài 4
def rotations_sliding2(s, k):
    n = len(s)
    score = sum(a == b for a, b in zip(s, s[1:]))
    answer = int(score == k)
    for p in range(n - 1):
        score -= s[p] == s[p + 1]
        score += s[p - 1] == s[p]
        answer += score == k
    return answer

for s, k, expected in examples:
    assert rotations_sliding2(s, k) == expected

# Bài 5(a)
def score_of(s, p):
    r = s[p:] + s[:p]
    return sum(a == b for a, b in zip(r, r[1:]))

def best_rotation(s):
    n = len(s)
    return max(range(n), key=lambda p: (score_of(s, p), -p))

assert best_rotation("aab") == 0
assert best_rotation("abca") == 1
assert best_rotation("aaaa") == 0

# Bài 5(b)
def count_distinct_rotations(s, k):
    n = len(s)
    rotations = {s[p:] + s[:p] for p in range(n)}
    return sum(sum(a == b for a, b in zip(r, r[1:])) == k for r in rotations)

assert count_distinct_rotations("aa", 1) == 1
assert count_distinct_rotations("abab", 0) == 2
assert count_distinct_rotations("aab", 1) == 2
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (xoay vòng + cặp liền kề trên vòng tròn)

```python
class Solution:
    def countRotations(self, s: str, k: int) -> int:
        n = len(s)
        cyc = [1 if s[i] == s[(i + 1) % n] else 0 for i in range(n)]
        total = sum(cyc)
        return sum(1 for p in range(n) if total - cyc[(p - 1) % n] == k)
```

**Checklist**

- [ ] Hiểu "phép xoay `p`" là `s[p:] + s[:p]` và có đúng `n` phép (`p = 0..n-1`)?
- [ ] Điểm chỉ tính `n - 1` cặp liền kề của **chuỗi thẳng**, **không** tính cặp đuôi–đầu?
- [ ] Công thức `điểm(p) = T − cyc[(p−1) % n]` với `T` là số cặp bằng nhau **trên vòng tròn** (`n` cặp)?
- [ ] Không gộp các phép xoay cho cùng chuỗi (đề đếm theo phép xoay)?
- [ ] Đã thử: `n = 2`, toàn ký tự giống nhau, xen kẽ hoàn toàn, `k = 0` và `k = n - 1`?
- [ ] Đã đối chiếu với brute force `s[p:] + s[:p]` trên chuỗi ngẫu nhiên nhỏ?
- [ ] Skeleton đúng `class Solution` / `def countRotations(self, s: str, k: int) -> int`?